# HPC Assignment 8 – Performance Analysis Graphs
## Group 3 | Parallel Interpolation with Particle Mover (MPI + OpenMP)

### Serial Baseline Estimation
Since `mpi=1, omp=1` was not explicitly measured, the serial baseline is estimated as **1.08×** the `mpi=1, omp=2` total time.  
This accounts for the marginal (and sometimes negligible) speedup that 2 OMP threads provide for these problem sizes.  
The factor 1.08 is conservative — you should **replace these values** with real `mpi=1, omp=1` measurements if available.

| Config | Grid      | Points | Serial Estimate (s) |
|--------|-----------|--------|---------------------|
| A      | 250×100   | 0.9M   | 0.523               |
| B      | 250×100   | 5M     | 2.860               |
| C      | 500×200   | 3.6M   | 5.115               |
| D      | 500×200   | 20M    | 14.094              |
| E      | 1000×400  | 14M    | 10.550              |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 9,
    'lines.linewidth': 2,
    'lines.markersize': 7,
    'axes.grid': True,
    'grid.alpha': 0.4,
})

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CONFIGURATION – update serial baselines with real values if available
# ─────────────────────────────────────────────────────────────────────

CSV_FILES = {
    'A': 'timings_a.csv',
    'B': 'timings_b.csv',
    'C': 'timings_c.csv',
    'D': 'timings_d.csv',
    'E': 'timings_e.csv',
}

CONFIG_LABELS = {
    'A': 'Config A: Nx=250, Ny=100, N=0.9M pts',
    'B': 'Config B: Nx=250, Ny=100, N=5M pts',
    'C': 'Config C: Nx=500, Ny=200, N=3.6M pts',
    'D': 'Config D: Nx=500, Ny=200, N=20M pts',
    'E': 'Config E: Nx=1000, Ny=400, N=14M pts',
}

# Serial baselines (mpi=1, omp=1). Replace with measured values if available.
SERIAL_BASELINES = {
    'A': 0.523,
    'B': 2.860,
    'C': 5.115,
    'D': 14.094,
    'E': 10.550,
}

COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
CORE_COUNTS = [2, 4, 8, 16, 32, 64]

In [ ]:
def load_and_process(csv_path, serial_baseline):
    """Load CSV, add cores column, compute best time per core count, speedup, efficiency."""
    df = pd.read_csv(csv_path)
    df['cores'] = df['mpi'] * df['omp']

    # Best (minimum) total time per core count
    best = df.groupby('cores')['total'].min().reset_index()
    best.columns = ['cores', 'best_total']

    # Corresponding best row details for phase breakdown
    best_rows = []
    for _, row in best.iterrows():
        c = row['cores']
        subset = df[df['cores'] == c]
        best_row = subset.loc[subset['total'].idxmin()].copy()
        best_rows.append(best_row)
    best_df = pd.DataFrame(best_rows).reset_index(drop=True)
    best_df = best_df[best_df['cores'].isin(CORE_COUNTS)].copy()

    best_df['speedup']    = serial_baseline / best_df['best_total'] if 'best_total' in best_df else serial_baseline / best_df['total']
    # recalculate from best_total
    best_df['speedup']    = serial_baseline / best_df['total']
    best_df['efficiency'] = best_df['speedup'] / best_df['cores'] * 100

    # Best MPI x OMP label
    best_df['config_label'] = best_df.apply(lambda r: f"{int(r['mpi'])}×{int(r['omp'])}", axis=1)

    return df, best_df

# Load all configs
data = {}
for key, path in CSV_FILES.items():
    raw, best = load_and_process(path, SERIAL_BASELINES[key])
    data[key] = {'raw': raw, 'best': best}
    print(f"Config {key} loaded. Core counts: {sorted(best['cores'].tolist())}")

## 1. Execution Time vs Cores – All Configurations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for idx, (key, color) in enumerate(zip(data.keys(), COLORS)):
    ax = axes[idx]
    best = data[key]['best']

    ax.plot(best['cores'], best['total'], marker='o', color=color, label='Best parallel time')
    ax.axhline(y=SERIAL_BASELINES[key], color='gray', linestyle='--', linewidth=1.5, label=f'Serial ({SERIAL_BASELINES[key]:.3f}s)')

    for _, row in best.iterrows():
        ax.annotate(row['config_label'],
                    (row['cores'], row['total']),
                    textcoords='offset points', xytext=(4, 4), fontsize=7.5, color='#333333')

    ax.set_title(CONFIG_LABELS[key], fontsize=10, fontweight='bold')
    ax.set_xlabel('Number of Cores')
    ax.set_ylabel('Execution Time (s)')
    ax.set_xscale('log', base=2)
    ax.set_xticks(CORE_COUNTS)
    ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
fig.suptitle('Execution Time vs Number of Cores (MPI+OpenMP)\nAnnotations show best MPI×OMP split', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('time_vs_cores.png', bbox_inches='tight')
plt.show()
print('Saved: time_vs_cores.png')

## 2. Speedup vs Cores – All Configurations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

ideal_cores = np.array([1] + CORE_COUNTS)

for idx, (key, color) in enumerate(zip(data.keys(), COLORS)):
    ax = axes[idx]
    best = data[key]['best']

    ax.plot(best['cores'], best['speedup'], marker='o', color=color, label='Achieved speedup')
    ax.plot(ideal_cores, ideal_cores, 'k--', linewidth=1.2, label='Ideal (linear)')

    for _, row in best.iterrows():
        ax.annotate(f"{row['speedup']:.2f}x",
                    (row['cores'], row['speedup']),
                    textcoords='offset points', xytext=(4, 3), fontsize=7.5, color='#333333')

    ax.set_title(CONFIG_LABELS[key], fontsize=10, fontweight='bold')
    ax.set_xlabel('Number of Cores')
    ax.set_ylabel('Speedup')
    ax.set_xscale('log', base=2)
    ax.set_xticks(CORE_COUNTS)
    ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
fig.suptitle('Speedup vs Number of Cores (MPI+OpenMP)\nDashed line = ideal linear speedup', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('speedup_vs_cores.png', bbox_inches='tight')
plt.show()
print('Saved: speedup_vs_cores.png')

## 3. Parallel Efficiency vs Cores – All Configurations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for idx, (key, color) in enumerate(zip(data.keys(), COLORS)):
    ax = axes[idx]
    best = data[key]['best']

    ax.plot(best['cores'], best['efficiency'], marker='s', color=color, label='Parallel efficiency')
    ax.axhline(y=100, color='k', linestyle='--', linewidth=1.2, label='100% efficiency')
    ax.axhline(y=50, color='orange', linestyle=':', linewidth=1.2, label='50% threshold')

    for _, row in best.iterrows():
        ax.annotate(f"{row['efficiency']:.1f}%",
                    (row['cores'], row['efficiency']),
                    textcoords='offset points', xytext=(4, 3), fontsize=7.5, color='#333333')

    ax.set_title(CONFIG_LABELS[key], fontsize=10, fontweight='bold')
    ax.set_xlabel('Number of Cores')
    ax.set_ylabel('Parallel Efficiency (%)')
    ax.set_xscale('log', base=2)
    ax.set_xticks(CORE_COUNTS)
    ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
    ax.set_ylim(0, 120)
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
fig.suptitle('Parallel Efficiency vs Number of Cores (MPI+OpenMP)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('efficiency_vs_cores.png', bbox_inches='tight')
plt.show()
print('Saved: efficiency_vs_cores.png')

## 4. Phase-wise Time Breakdown (Stacked Bar) – Best Config per Core Count

In [ ]:
phases       = ['interp', 'reduce', 'broadcast', 'norm', 'mover', 'denorm']
phase_labels = ['Interpolation', 'MPI Reduce', 'MPI Broadcast', 'Normalize', 'Mover', 'Denormalize']
phase_colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f', '#edc948']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, key in enumerate(data.keys()):
    ax = axes[idx]
    best = data[key]['best']

    x = np.arange(len(best))
    bar_width = 0.6
    bottoms = np.zeros(len(best))

    for phase, label, color in zip(phases, phase_labels, phase_colors):
        values = best[phase].values
        ax.bar(x, values, bottom=bottoms, label=label, color=color, width=bar_width, edgecolor='white', linewidth=0.5)
        bottoms += values

    ax.set_xticks(x)
    ax.set_xticklabels([f"{int(r['cores'])}c\n{r['config_label']}" for _, r in best.iterrows()], fontsize=8)
    ax.set_title(CONFIG_LABELS[key], fontsize=10, fontweight='bold')
    ax.set_xlabel('Cores (MPI×OMP)')
    ax.set_ylabel('Time (s)')
    if idx == 0:
        ax.legend(loc='upper right', fontsize=7.5, ncol=2)

axes[-1].set_visible(False)
fig.suptitle('Phase-wise Execution Time Breakdown\n(Best MPI×OMP combination per core count)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('phase_breakdown.png', bbox_inches='tight')
plt.show()
print('Saved: phase_breakdown.png')

## 5. Interpolation vs Mover Phase Comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for idx, (key, color) in enumerate(zip(data.keys(), COLORS)):
    ax = axes[idx]
    best = data[key]['best']

    ax.plot(best['cores'], best['interp'], marker='o', color='#4e79a7', label='Interpolation', linewidth=2)
    ax.plot(best['cores'], best['mover'],  marker='s', color='#59a14f', label='Mover',         linewidth=2)
    ax.plot(best['cores'], best['reduce'], marker='^', color='#f28e2b', label='MPI Reduce',    linewidth=1.5, linestyle='--')
    ax.plot(best['cores'], best['broadcast'], marker='v', color='#e15759', label='MPI Bcast', linewidth=1.5, linestyle='--')

    ax.set_title(CONFIG_LABELS[key], fontsize=10, fontweight='bold')
    ax.set_xlabel('Number of Cores')
    ax.set_ylabel('Time (s)')
    ax.set_xscale('log', base=2)
    ax.set_xticks(CORE_COUNTS)
    ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
fig.suptitle('Interpolation vs Mover Phase Time (Best MPI×OMP per Core Count)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('interp_vs_mover.png', bbox_inches='tight')
plt.show()
print('Saved: interp_vs_mover.png')

## 6. Combined Speedup Comparison – All 5 Configurations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for key, color in zip(data.keys(), COLORS):
    best = data[key]['best']
    ax.plot(best['cores'], best['speedup'], marker='o', color=color, label=CONFIG_LABELS[key])

ax.plot(ideal_cores, ideal_cores, 'k--', linewidth=1.5, label='Ideal linear speedup')
ax.set_xscale('log', base=2)
ax.set_xticks(CORE_COUNTS)
ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
ax.set_xlabel('Number of Cores')
ax.set_ylabel('Speedup')
ax.set_title('Speedup vs Cores – All Configurations Comparison', fontweight='bold')
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.savefig('speedup_comparison.png', bbox_inches='tight')
plt.show()
print('Saved: speedup_comparison.png')

## 7. Best MPI×OMP Heatmap per Config and Core Count

In [ ]:
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for idx, key in enumerate(data.keys()):
    ax = axes[idx]
    raw = data[key]['raw']
    raw_f = raw[raw['cores'].isin(CORE_COUNTS)].copy()

    # For each core count, find the best mpi,omp split
    records = []
    for c in CORE_COUNTS:
        subset = raw_f[raw_f['cores'] == c]
        if len(subset) == 0:
            continue
        best_idx = subset['total'].idxmin()
        best_row = subset.loc[best_idx]
        records.append({'cores': c, 'mpi': int(best_row['mpi']), 'omp': int(best_row['omp']), 'total': best_row['total']})

    rec_df = pd.DataFrame(records)

    # Build pivot: rows=unique mpi, cols=unique omp, values=total (NaN if not run)
    pivot = raw_f.pivot_table(index='mpi', columns='omp', values='total', aggfunc='min')
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn_r')

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns.tolist(), fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index.tolist(), fontsize=8)
    ax.set_xlabel('OMP Threads', fontsize=9)
    ax.set_ylabel('MPI Processes', fontsize=9)
    ax.set_title(f'Config {key}\n(Min Time, s)', fontsize=9, fontweight='bold')

    # Annotate each cell
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6.5,
                        color='black')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('MPI × OMP Configuration Heatmap – Minimum Execution Time (s)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('mpi_omp_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved: mpi_omp_heatmap.png')

## 8. Summary Table – Best Results per Configuration

In [ ]:
summary_rows = []
for key in data.keys():
    best = data[key]['best']
    max_su_row = best.loc[best['speedup'].idxmax()]
    summary_rows.append({
        'Config': key,
        'Description': CONFIG_LABELS[key].split(': ')[1],
        'Serial Time (s)': SERIAL_BASELINES[key],
        'Best Parallel Time (s)': round(max_su_row['total'], 4),
        'Best MPI×OMP': max_su_row['config_label'],
        'Cores': int(max_su_row['cores']),
        'Max Speedup': round(max_su_row['speedup'], 2),
        'Efficiency at Max Speedup (%)': round(max_su_row['efficiency'], 1),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv('summary_results.csv', index=False)
print('\nSaved: summary_results.csv')